# AWS SageMaker, Part 1. A Private Platform and the Data

Building a SageMaker setup the way a company would build it, in a private network, then connecting VS Code to it and getting a messy CSV ready for training. The next class trains a model on this data and serves it three ways.

---

**How to use this notebook**

Every command, policy, file body, and value intended for copying is in an **actual code cell**, with its destination labeled immediately above it. Select and copy directly from the visible cell; the instructions and screenshots stay rendered. Plain names and paths are intentional code cells.

Use this notebook as a copy-and-paste guide. Run commands only in the labeled terminal, and paste policies or values into their labeled fields. Do not use **Run all** here. Sample outputs remain in Markdown after the corresponding code or command.

## **Agenda**

1. [Where SageMaker fits](#scrollTo=c003)
2. [What this costs, and what keeps billing](#scrollTo=c004)
3. [S3, the place everything meets](#scrollTo=c005)
4. [A private network for SageMaker](#scrollTo=c006)
5. [A role, a domain, and VS Code in a space](#scrollTo=c007)
6. [A first look at the data](#scrollTo=c008)
7. [Data Wrangler in Canvas](#scrollTo=c013)
8. [Getting the data into the shape XGBoost expects](#scrollTo=c014)
9. [Shutting down until next class](#scrollTo=c017)
10. [Class summary](#scrollTo=c018)
11. [References and further reading](#scrollTo=c019)
12. [What's next](#scrollTo=c020)

## **Preface**

So far every model in this course has been trained on your own machine. Then you shipped it yourself. You put it in a container, ran the container on ECS, and wired up the deploy with GitHub Actions. That works, and you now know every piece of it. But you also wrote every piece of it. That includes the Dockerfile, the FastAPI app, the load balancer, the health check and the scaling.

SageMaker is AWS's managed service for machine learning. Its pitch is that you hand over the parts that are the same for every model. You give it data in S3 and pick an algorithm or a container. It rents a machine to train on and throws the machine away when training ends. It puts the model behind an HTTPS endpoint with logs and metrics already wired up.

SageMaker takes two classes, and they match two different jobs in a real team. In this class you are mostly the platform engineer. You build the network, the role and the SageMaker domain in the console, the way a company with customer data would. Then you switch to the ML engineer's seat. You open VS Code, connect it to a machine inside that network, and look at the data in Python files. In the next class, the same VS Code window trains the model and serves it, all from code.

The project is an online shop that wants to know which orders will be sent back. Section 1.1 introduces it before any AWS work starts.

**[Instructor note]** Sections 2 to 5 happen in the browser, in the **AWS console**, signed in with the region set to **N. Virginia (us-east-1)** at the top right. Fill each form live and explain the fields as you go. One step in section 4 runs in **CloudShell**, the terminal built into the console. Section 5.4 is a setup on your own laptop, best done before class. From section 5.5 on, the work happens in **VS Code on your laptop, connected to the SageMaker space**. Each Python file goes in a folder called `returns-ml` in the space's home folder, and each one is run from **VS Code's terminal**, which is a terminal on the space, with `python <file>.py`. Section 7 is back in the browser, in Canvas. Outputs below come from a real run, so your timings will differ a little. Section 9 stops everything that bills by the hour. Do not end the class without it.

**By the end you will be able to do all of this.**

1. Explain what SageMaker manages for you, and how teams split the work between the console and code.
2. Name what bills by the hour in this setup and stop each one.
3. Create an S3 bucket and lay out its folders.
4. Build a private VPC for SageMaker with a NAT gateway and the endpoints it needs, and explain what each piece is for.
5. Create a role that can use your bucket, set up a VPC-only domain, and connect VS Code to a space inside it.
6. Read data from S3 in Python and catch a leaky column before it fools you.
7. Profile the same data in Data Wrangler and read its target leakage warning.
8. Prepare a CSV in the exact format the built-in XGBoost algorithm expects.

**Prerequisites.** Pandas and scikit-learn. A train and validation split and what AUC means. The AWS console, IAM roles, regions, VPCs, subnets and security groups from the ECS session. VS Code on your laptop. An AWS account where you can create resources.

## **1. Where SageMaker fits**

Think back to the ECS session. You built a machine to run the model, a network, an address and something to watch it. SageMaker gives you the same things for machine learning, already joined together.

![sagemaker_flow.png](https://d2beiqkhq929f0.cloudfront.net/public_assets/assets/000/244/510/original/sagemaker_flow.png?1791270275)

Everything goes through **S3**. Your code writes training files there. A **training job** reads them, trains on a machine that exists only for those few minutes, and writes the model back to S3 as one file called `model.tar.gz`. An **endpoint** loads that file and answers requests over HTTPS. A **batch transform job** loads the same file, scores a whole folder of rows and writes the scores back to S3. Nothing talks to anything else directly. If you know where each thing reads and writes in S3, you can debug most of SageMaker. This class gets everything up to the left side of that picture ready. The next class does the rest.

SageMaker has dozens of features, and teams do not use them equally. The parts that carry most of the weight are these four.

- **Training jobs**, because renting a large machine for twenty minutes beats owning one.
- **Endpoints**, because they come with HTTPS, logs, metrics and autoscaling without writing a server.
- **Pipelines**, which chain those steps together into one repeatable run.
- **The model registry**, which keeps track of which model version is live.

All four are normally driven from code, not from console forms. Code can be reviewed, kept in Git and run again next month with one change. A form you filled in by hand cannot. So you click through the console only where you are setting up the platform, and you write Python for everything that touches the data and the model. The low-code tools, Canvas and Data Wrangler, are aimed mostly at analysts. Section 7 covers Data Wrangler and is honest about where it fits.

**[Instructor note]** Browser. Type `SageMaker` in the console search bar and open **Amazon SageMaker AI**. Show the left menu, with Domains, Studio, Canvas, Training jobs and Inference. Nothing is created here.

**AWS console search**

In [ ]:
SageMaker

### 1.1 The project, predicting returns

The project for both SageMaker classes is an online shop. It sells clothes, shoes, electronics, home goods, beauty products and books. More than one order in five comes back. Every return costs the shop shipping in both directions, and the item often cannot be sold again at full price. If the shop knew at checkout which orders were likely to come back, it could act early. It could show a size guide, ask the customer to confirm a cash on delivery order, or plan stock for the items that will return.

So the model answers one question. **When an order is placed, how likely is it to be returned?** The label is yes or no, so this is binary classification. The model's output is a probability between 0 and 1.

The data is one CSV file, [returns.csv](https://d2beiqkhq929f0.cloudfront.net/public_assets/assets/000/244/530/original/returns.csv?1791270327), with 12,000 past orders and one row per order. It is synthetic. A script generated it from known rules, so we can check that each tool finds what is really there. It is shaped like an export from a shop's order database, mess included. It has a header row, text categories, some missing values and the label in the last column.

| Column | What it holds |
|---|---|
| `order_id`, `order_date` | Which order this is and when it was placed |
| `category` | apparel, footwear, electronics, home, beauty or books |
| `price`, `discount_pct` | The item's price and the discount on it |
| `delivery_days` | Delivery time in days |
| `customer_tenure_days` | How long the customer has had an account |
| `prior_orders`, `prior_returns` | The customer's history with the shop |
| `payment_method` | card, upi, cod (cash on delivery) or wallet |
| `is_gift` | 1 if the order was marked as a gift |
| `seller_rating` | The seller's rating out of 5 |
| `refund_amount` | The amount refunded on the order |
| `returned` | The label, 1 if the order came back and 0 if not |

In this class you upload the file to S3, look at it from VS Code, profile it in Data Wrangler, and turn it into the format the built-in XGBoost algorithm wants. In the next class a training job trains the model, and the same model is served three ways. Look over the columns before you start. One of them causes trouble, and section 6 shows which.

## **2. What this costs, and what keeps billing**

SageMaker bills by the second for any machine it runs for you. The trap is that some of those machines keep running after you stop looking at them. Here are the prices for what the two SageMaker classes use in us-east-1, from the AWS price list in October 2026.

| What runs | Size | Price |
|---|---|---|
| JupyterLab space | ml.m5.large | 0.115 USD per hour while the space is running |
| Canvas, including Data Wrangler | Workspace | 1.90 USD per hour until you log out |
| NAT gateway | One, regional | 0.045 USD per hour, plus 0.045 USD per GB through it |
| VPC interface endpoints | 6 endpoints in 2 zones | 0.01 USD per hour each, per zone, so 0.12 USD per hour |
| Training job (next class) | ml.m5.large | 0.115 USD per hour, only while the job runs |
| Real-time endpoint (next class) | ml.m5.large | 0.115 USD per hour, all day, even with no requests |
| Serverless endpoint (next class) | 1 GB of memory | Charged per request for the time it spends working |

Training jobs and batch jobs stop by themselves. Nothing else on that list does.

- **A JupyterLab space bills while it runs**, whether or not anyone is connected to it.
- **Canvas bills until you log out of it.** Closing the browser tab is not logging out. Left open, it costs about 1,400 USD a month.
- **The NAT gateway and the interface endpoints bill every hour they exist**, even when nothing uses them. Together they cost about 120 USD a month if they are forgotten. That is why section 9 deletes them tonight and the next class brings them back with one script.
- **A real-time endpoint bills every hour it exists.** One left running costs about 84 USD a month.

This class, done once, costs about one US dollar, and the biggest single part is Canvas. Set a budget alert before starting, the way the ECS session did, because the alert is the only thing that will tell you about something you forgot.

## **3. S3, the place everything meets**

S3 stores files, which AWS calls **objects**, in **buckets**. A bucket name is unique across all of AWS, not just your account. So you cannot call yours `returns`, because somebody else already has. The usual fix is a name with your initials or a date in it.

Inside a bucket there are no real folders. An object's name can contain slashes, as in `train/train.csv`, and the console shows the part before the slash as a folder. That part is called a **prefix**. SageMaker asks for prefixes all the time. When a training job is told to read `s3://bucket/train/`, it reads every object whose name starts with `train/`.

Both classes use one bucket with these prefixes.

| Prefix | Holds | Written by |
|---|---|---|
| `raw/` | The orders CSV as it arrives | You, in the console |
| `train/`, `validation/` | Training files in XGBoost format | `prepare.py`, in this class |
| `batch-input/`, `batch-labels/` | Rows to score, and their answers kept aside | `prepare.py`, in this class |
| `output/` | The trained model, `model.tar.gz` | The training job, next class |
| `batch-output/` | Scores for those rows | The batch transform job, next class |

### 3.1 Create the bucket and upload the data

**[Instructor note]** Browser. Search for `S3`, open it and click **Create bucket**. Leave the bucket type as **General purpose** and the namespace as **Global namespace**. Type a name such as `returns-ml-` followed by your initials and today's date. Leave everything else as it is and click **Create bucket** at the bottom.

**AWS console search**

In [ ]:
S3

**Bucket name prefix — append your initials and today's date to make a unique name**

In [ ]:
returns-ml-

![s3_create_bucket.png](https://d2beiqkhq929f0.cloudfront.net/public_assets/assets/000/244/983/original/s3_create_bucket.png?1791359513)

Block Public Access stays on. Nothing in these classes needs a public bucket. SageMaker reads it through its role, and section 5.1 gives that role this bucket by name.

**[Instructor note]** Browser. First download [returns.csv](https://d2beiqkhq929f0.cloudfront.net/public_assets/assets/000/244/530/original/returns.csv?1791270327) to your machine. Open the new bucket, click **Create folder**, name it `raw` and create it. Open `raw/`, click **Upload**, then **Add files**, pick `returns.csv` and click **Upload**.

**S3 folder name**

In [ ]:
raw

![s3_upload_done.png](https://d2beiqkhq929f0.cloudfront.net/public_assets/assets/000/244/984/original/s3_upload_done.png?1791359515)

![s3_bucket_raw.png](https://d2beiqkhq929f0.cloudfront.net/public_assets/assets/000/244/982/original/s3_bucket_raw.png?1791359510)

The file is 780.7 KB and 12,000 rows. Copy the bucket name somewhere. The code asks for it once, in `config.py`.

## **4. A private network for SageMaker**

In the ECS session you built a VPC with public subnets, because the load balancer had to be reachable from the internet. Here nothing needs to be reachable from outside. Companies that hold customer data put their ML machines in **private subnets**, where nothing on the internet can start a connection to them. SageMaker calls a domain set up this way **VPC only**.

The machines still need to get out in two ways. They need AWS services, such as S3, SageMaker itself, and the endpoints you will call next class. And they need some of the public internet, for things like `pip install` and the small server that VS Code installs when it connects. This section builds both ways out.

- **VPC endpoints** are private doors from your VPC to one AWS service each. Traffic through them never leaves AWS's network.
- **A NAT gateway** lets machines in private subnets start connections out to the internet, while nothing outside can start a connection in. Think of a one-way door.

### 4.1 The VPC and the NAT gateway

**[Instructor note]** Browser. Search for `VPC`, open **Your VPCs** and click **Create VPC**. Choose **VPC and more** and type `returns-ml` as the name tag. Keep the CIDR block `10.0.0.0/16` and **2** availability zones. Open **Customize AZs** and pick **use1-az2** and **use1-az4**. Set **public subnets** to **0** and keep **private subnets** at **2**. Set **NAT gateways** to **Regional**. Keep **VPC endpoints** at **S3 Gateway** and both DNS boxes ticked. Click **Create VPC**. The NAT gateway takes a couple of minutes, so explain the preview while it builds.

**AWS console search**

In [ ]:
VPC

**VPC name tag**

In [ ]:
returns-ml

**IPv4 CIDR block**

In [ ]:
10.0.0.0/16

**First availability zone ID**

In [ ]:
use1-az2

**Second availability zone ID**

In [ ]:
use1-az4

![vpc_form_top.png](https://d2beiqkhq929f0.cloudfront.net/public_assets/assets/000/245/000/original/vpc_form_top.png?1791359551)

![vpc_form_subnets.png](https://d2beiqkhq929f0.cloudfront.net/public_assets/assets/000/244/999/original/vpc_form_subnets.png?1791359548)

![vpc_form_endpoints.png](https://d2beiqkhq929f0.cloudfront.net/public_assets/assets/000/244/998/original/vpc_form_endpoints.png?1791359547)

![vpc_preview.png](https://d2beiqkhq929f0.cloudfront.net/public_assets/assets/000/245/001/original/vpc_preview.png?1791359553)

The preview shows what this one form builds, and every piece has a job.

- **Two private subnets**, one in each zone. Nothing on the internet can reach a machine in them.
- **A regional NAT gateway** and an **internet gateway** behind it. The regional kind is new. It needs no public subnet of your own and covers both zones by itself.
- **The S3 Gateway endpoint.** This one is free. It adds a route for S3 to the private subnets, so S3 traffic never goes through the NAT.
- **The zones.** The form suggests the first two zones. One endpoint used later does not exist in `use1-az1`, so this setup uses `use1-az2` and `use1-az4`. Zone names like `us-east-1b` point to different zone IDs in different accounts, so go by the ID in brackets.

Both DNS boxes matter too. The endpoints in 4.3 work by changing what AWS's normal service addresses point to inside the VPC, and that needs both.

![vpc_created.png](https://d2beiqkhq929f0.cloudfront.net/public_assets/assets/000/244/997/original/vpc_created.png?1791359545)

The form took 154 seconds here, most of it waiting for the NAT gateway. Each private subnet's route table now has three routes. Its own addresses stay `local`, S3 goes to the S3 endpoint, and everything else, `0.0.0.0/0`, goes to the NAT gateway.

### 4.2 Two security groups

You need two security groups, in the same pattern as the load balancer and task groups in the ECS session.

- `returns-studio-sg` goes on the SageMaker apps. AWS asks for TCP between the apps in this group, on ports 8192 and up, which JupyterLab's parts use to talk to each other. So this group lets in all TCP from itself and nothing else.
- `returns-endpoints-sg` goes on the VPC endpoints. It lets in HTTPS, port 443, only from the studio group.

**[Instructor note]** Browser. In the VPC console open **Security groups → Create security group**. Name it `returns-studio-sg`, give it a description, pick `returns-ml-vpc` and add no inbound rules yet. Click **Create security group**. A group cannot name itself as a source until it exists, so open the new group, click **Edit inbound rules**, add a rule of type **All TCP**, and in the source box type `returns-studio` and pick the group itself. Click **Save rules**.

**Studio security group name**

In [ ]:
returns-studio-sg

**VPC name to find in the selector**

In [ ]:
returns-ml-vpc

**Inbound-rule source search — select the studio group itself**

In [ ]:
returns-studio

![sg_studio_form.png](https://d2beiqkhq929f0.cloudfront.net/public_assets/assets/000/244/986/original/sg_studio_form.png?1791359519)

![sg_studio_rule.png](https://d2beiqkhq929f0.cloudfront.net/public_assets/assets/000/244/987/original/sg_studio_rule.png?1791359522)

**[Instructor note]** Browser. Create a second group, `returns-endpoints-sg`, in the same VPC. Add one inbound rule of type **HTTPS**, with `returns-studio-sg` as the source. Click **Create security group**.

**Endpoint security group name**

In [ ]:
returns-endpoints-sg

**HTTPS inbound-rule source — select this security group**

In [ ]:
returns-studio-sg

![sg_endpoints_form.png](https://d2beiqkhq929f0.cloudfront.net/public_assets/assets/000/244/985/original/sg_endpoints_form.png?1791359517)

### 4.3 The endpoints

An **interface endpoint** is a network card that AWS places in your subnets and that stands in for one service. With **private DNS** turned on, the service's normal address, such as `api.sagemaker.us-east-1.amazonaws.com`, points to that card when asked from inside the VPC. So boto3 and Studio need no change at all. They call the usual address and the traffic stays inside AWS.

With a NAT gateway in place, the machines could reach these services through the internet instead. Teams still add endpoints for the services that carry their data and their control traffic, so that traffic never crosses the internet at all.

| Endpoint | Who calls it | Why it is needed |
|---|---|---|
| `s3` (Gateway, free) | Your code and Canvas | Reading and writing the bucket |
| `sagemaker.api` | Studio and your code | Starting spaces and jobs, every SageMaker call |
| `sagemaker.runtime` | Your code | Calling your endpoints next class |
| `sts` and `logs` | The SageMaker Python SDK | Getting credentials and reading job logs |
| `ssm` and `ssmmessages` | Remote access from VS Code | The tunnel between your laptop and the space |

Interface endpoints are not free. Each costs 0.01 USD an hour in each zone, so these six cost 0.12 USD an hour, every hour they exist. Section 9 deletes them.

**[Instructor note]** Browser. In the VPC console open **Endpoints → Create endpoint**. Name it `returns-ml-sagemaker-api`, keep **AWS services** and search for `sagemaker.api`. Pick `com.amazonaws.us-east-1.sagemaker.api`, not the `-fips` one. Pick `returns-ml-vpc` and check that **Enable private DNS name** is ticked. Tick both zones that have a subnet and pick the subnet in each. Under security groups, tick only `returns-endpoints-sg`. Keep the policy at **Full access** and click **Create endpoint**.

**Endpoint name**

In [ ]:
returns-ml-sagemaker-api

**Endpoint service search**

In [ ]:
sagemaker.api

**Exact endpoint service name**

In [ ]:
com.amazonaws.us-east-1.sagemaker.api

**VPC name to find in the selector**

In [ ]:
returns-ml-vpc

**Security group to select**

In [ ]:
returns-endpoints-sg

![vpce_form_service.png](https://d2beiqkhq929f0.cloudfront.net/public_assets/assets/000/244/744/original/vpce_form_service.png?1791291660)

![vpce_form_network.png](https://d2beiqkhq929f0.cloudfront.net/public_assets/assets/000/245/003/original/vpce_form_network.png?1791359558)

![vpce_form_subnets.png](https://d2beiqkhq929f0.cloudfront.net/public_assets/assets/000/245/005/original/vpce_form_subnets.png?1791359563)

![vpce_form_sg.png](https://d2beiqkhq929f0.cloudfront.net/public_assets/assets/000/245/004/original/vpce_form_sg.png?1791359561)

That is one endpoint. The other five are the same form with a different service name. Teams do not click through that five times. They write it once as code and run it. **CloudShell** is a terminal inside the console. It is already signed in as you, has the AWS CLI installed, and costs nothing.

The script looks up the VPC, the subnets and the endpoint group by their names, so you do not have to copy any IDs. Then it creates one endpoint per service in a loop. Each one comes back as `pending` and turns `available` in a minute or two.

**[Instructor note]** Browser, CloudShell. Click the **CloudShell** icon at the bottom left of the console, wait for the prompt, then paste this whole block and press Enter. It prints one line for the IDs it found, then one line per endpoint.

**Copy the whole script → AWS CloudShell**

In [ ]:
VPC=$(aws ec2 describe-vpcs --filters Name=tag:Name,Values=returns-ml-vpc --query 'Vpcs[0].VpcId' --output text)
SUBNETS=$(aws ec2 describe-subnets --filters Name=vpc-id,Values=$VPC --query 'Subnets[].SubnetId' --output text)
SG=$(aws ec2 describe-security-groups --filters Name=vpc-id,Values=$VPC Name=group-name,Values=returns-endpoints-sg --query 'SecurityGroups[0].GroupId' --output text)
echo "VPC $VPC  subnets $SUBNETS  security group $SG"

for SERVICE in sagemaker.runtime sts logs ssm ssmmessages; do
  aws ec2 create-vpc-endpoint --vpc-id $VPC --vpc-endpoint-type Interface \
    --service-name com.amazonaws.us-east-1.$SERVICE --subnet-ids $SUBNETS \
    --security-group-ids $SG --private-dns-enabled \
    --tag-specifications "ResourceType=vpc-endpoint,Tags=[{Key=Name,Value=returns-ml-$SERVICE}]" \
    --query 'VpcEndpoint.[ServiceName,State]' --output text
done


Output from the run.

```
VPC vpc-0cea598e1340d89a8  subnets subnet-0302f9421c671af25	subnet-0d54f5480d6e718d0  security group sg-074103d099031880f
com.amazonaws.us-east-1.sagemaker.runtime	pending
com.amazonaws.us-east-1.sts	pending
com.amazonaws.us-east-1.logs	pending
com.amazonaws.us-east-1.ssm	pending
com.amazonaws.us-east-1.ssmmessages	pending
```

![vpce_list.png](https://d2beiqkhq929f0.cloudfront.net/public_assets/assets/000/245/006/original/vpce_list.png?1791359564)

All seven show **Available**, counting the S3 gateway. The network is ready, and nothing has been put in it yet.

### 4.4 Stricter networks

Some companies go further and give the ML machines no internet at all, with no NAT gateway. That works, but every service has to come through an endpoint, and the gaps are hard to find. While preparing this class, a domain with no NAT and only the SageMaker endpoints left Data Wrangler's list of data sources spinning for more than four minutes, with no error at all. It needed endpoints for Athena, Glue, EMR Serverless and Cognito, and Cognito is not even on AWS's own list for Canvas. VS Code's remote connection also needs the internet to install its server, so a fully closed network needs a pre-packaged copy of that server or a proxy. A common middle ground is to keep the NAT gateway and put an allow-list proxy or AWS Network Firewall in front of it, so the machines can reach only approved sites.

### **Checkpoint 4.1**

Read the network you just built. In the VPC console, open **Route tables** and open `returns-ml-rtb-private1-us-east-1b`, or whatever your first private table is called. Read its **Routes** tab. You should find three routes. One is `local`, one sends a `pl-` prefix list to the S3 endpoint, and one sends `0.0.0.0/0` to the NAT gateway. Then open the route table named `rtb-regional-nat` and find the route to the internet gateway. Draw the path a `pip install` takes from the space to the internet, and the path a call to S3 takes. They are different, and that difference is the reason the S3 endpoint exists.

**Example private route table name — your zone suffix may differ**

In [ ]:
returns-ml-rtb-private1-us-east-1b

**Regional NAT route table name**

In [ ]:
rtb-regional-nat

## **5. A role, a domain, and VS Code in a space**

Before you can run anything in SageMaker, you need a **domain**. A domain is SageMaker's container for users, their settings, their network and the role their work runs as. Inside it, each person has a **user profile**. Studio is the web app you open from that profile. A **space** is a named disk plus a machine that runs JupyterLab or Code Editor on top of it. You can stop the machine and keep the disk.

The console offers a one-click **quick setup** for all of this. It makes a role for you and drops the domain into the region's default VPC with internet access. That is fine for a personal account. This class uses **Set up for organizations** instead, because it lets you choose the role and the network yourself.

### 5.1 The role

Every Python file and every job in both classes acts as one IAM role. It has to be allowed to use SageMaker and to read and write your bucket. The managed policy `AmazonSageMakerFullAccess` covers SageMaker. Despite its name, it only lets a role use S3 buckets whose name contains `sagemaker`. Your bucket's name does not, and it does not have to. Teams add a small policy of their own that names the bucket.

**[Instructor note]** Browser. Search for `IAM`, open **Roles** and click **Create role**. Keep **AWS service**, pick **SageMaker** as the service and keep the use case **SageMaker - Execution**. Click **Next**. The permissions page shows `AmazonSageMakerFullAccess` already attached and does not let you add more here. Click **Next**, name the role `returns-ml-sagemaker-role` and click **Create role**.

**AWS console search**

In [ ]:
IAM

**IAM role name**

In [ ]:
returns-ml-sagemaker-role

![role_usecase.png](https://d2beiqkhq929f0.cloudfront.net/public_assets/assets/000/244/749/original/role_usecase.png?1791291672)

![role_permissions.png](https://d2beiqkhq929f0.cloudfront.net/public_assets/assets/000/244/979/original/role_permissions.png?1791359503)

![role_name.png](https://d2beiqkhq929f0.cloudfront.net/public_assets/assets/000/244/751/original/role_name.png?1791291677)

**[Instructor note]** Browser. Open the new role, click **Add permissions → Attach policies**, search for `AmazonSageMakerCanvas`, tick `AmazonSageMakerCanvasFullAccess` and `AmazonSageMakerCanvasDataPrepFullAccess`, and click **Add permissions**. Data Wrangler in section 7 needs both.

**Canvas policy search**

In [ ]:
AmazonSageMakerCanvas

**First managed policy to select**

In [ ]:
AmazonSageMakerCanvasFullAccess

**Second managed policy to select**

In [ ]:
AmazonSageMakerCanvasDataPrepFullAccess

![role_attach_canvas.png](https://d2beiqkhq929f0.cloudfront.net/public_assets/assets/000/244/977/original/role_attach_canvas.png?1791359499)

**[Instructor note]** Browser. On the same role, click **Add permissions → Create inline policy**, switch the editor to **JSON**, delete what is there and paste the policy below. Replace `YOUR-BUCKET-NAME` in both `Resource` lines with your own bucket's name, and keep the `/*` on the second one. Click **Next**, name it `returns-ml-bucket-access` and click **Create policy**.

**Copy JSON → IAM inline policy editor; replace YOUR-BUCKET-NAME in both Resource lines**

In [ ]:
{
  "Version": "2012-10-17",
  "Statement": [
    {
      "Sid": "SeeTheClassBucket",
      "Effect": "Allow",
      "Action": ["s3:ListBucket", "s3:GetBucketLocation"],
      "Resource": "arn:aws:s3:::YOUR-BUCKET-NAME"
    },
    {
      "Sid": "ReadAndWriteItsFiles",
      "Effect": "Allow",
      "Action": ["s3:GetObject", "s3:PutObject", "s3:DeleteObject"],
      "Resource": "arn:aws:s3:::YOUR-BUCKET-NAME/*"
    }
  ]
}


**Inline policy name**

In [ ]:
returns-ml-bucket-access

The two statements cover the two kinds of S3 resource. The first is the bucket itself, which is what listing its contents acts on. The second ends in `/*` and covers every file inside it. A policy with only the first one lets the role see `train.csv` but not download it. The next class shows a real training job that failed in exactly that way.

![role_inline_policy.png](https://d2beiqkhq929f0.cloudfront.net/public_assets/assets/000/244/978/original/role_inline_policy.png?1791359500)

The role needs one more permission, and it is easy to miss. Connecting VS Code to a space calls `sagemaker:StartSession`, and `AmazonSageMakerFullAccess` does not include it. While preparing this class, the connection failed for five minutes with only "Resource is not connected, please wait before retrying", and nothing pointed at the role. This is AWS's own policy for it. It lets a user open a session only on spaces that belong to their own user profile.

**[Instructor note]** Browser. Create a second inline policy the same way, paste the policy below unchanged, and name it `returns-ml-remote-access`. The editor shows a yellow warning on the condition line. The policy saves and works anyway.

**Copy JSON → IAM inline policy editor for remote access**

In [ ]:
{
  "Version": "2012-10-17",
  "Statement": [
    {
      "Sid": "RestrictStartSessionOnSpacesToUserProfile",
      "Effect": "Allow",
      "Action": ["sagemaker:StartSession"],
      "Resource": "arn:*:sagemaker:*:*:space/${sagemaker:DomainId}/*",
      "Condition": {
        "ArnLike": {
          "sagemaker:ResourceTag/sagemaker:user-profile-arn": "arn:aws:sagemaker:*:*:user-profile/${sagemaker:DomainId}/${sagemaker:UserProfileName}"
        }
      }
    }
  ]
}


**Inline policy name**

In [ ]:
returns-ml-remote-access

![role_remote_policy.png](https://d2beiqkhq929f0.cloudfront.net/public_assets/assets/000/244/980/original/role_remote_policy.png?1791359505)

![role_summary.png](https://d2beiqkhq929f0.cloudfront.net/public_assets/assets/000/244/981/original/role_summary.png?1791359507)

The role now has three AWS policies and two of your own. `AmazonSageMakerFullAccess` is still broad, since it can create any SageMaker resource in the account. A stricter team replaces it with a hand-written list of actions. That is more than these classes need, and the part that matters most, which data the role can touch, is already narrow.

### 5.2 The domain

**[Instructor note]** Browser. In SageMaker AI, open **Domains**, click **Create domain**, choose **Set up for organizations** and click **Set up**. The wizard has seven steps. Fill them as below and leave the rest as they are.

- **Step 1.** Domain name `returns-ml-domain`. Keep **Login through IAM**. Under IAM users click **Add user** and add a profile called `instructor`.

**Domain name**

In [ ]:
returns-ml-domain

**User profile name**

In [ ]:
instructor

- **Step 2.** Choose **Use an existing role** and pick `returns-ml-sagemaker-role`.

**Existing IAM role to select**

In [ ]:
returns-ml-sagemaker-role

- **Step 3.** Under JupyterLab, turn on **Enable idle shutdown** and set the time to **60** minutes.

**Idle shutdown timeout, in minutes**

In [ ]:
60

- **Step 4.** Click **Next**.
- **Step 5.** Keep **Virtual Private Cloud (VPC) only**. Pick `returns-ml-vpc`, both of its subnets and `returns-studio-sg`.

**Domain VPC to select**

In [ ]:
returns-ml-vpc

**Domain security group to select**

In [ ]:
returns-studio-sg

- **Step 6.** Check that **Enable Home EFS file system creation** is off.
- **Step 7.** Check the summary and click **Submit**.

![domain_step1.png](https://d2beiqkhq929f0.cloudfront.net/public_assets/assets/000/244/963/original/domain_step1.png?1791359467)

![domain_step2_role.png](https://d2beiqkhq929f0.cloudfront.net/public_assets/assets/000/244/964/original/domain_step2_role.png?1791359470)

In step 2, look at which role the box picked. Here it picked the right one, because it was the only SageMaker role in the account. In an earlier run in an account with older roles, it filled in a leftover `AmazonSageMaker-ExecutionRole-...` from a quick setup. That looks reasonable and is wrong.

![domain_step3_idle.png](https://d2beiqkhq929f0.cloudfront.net/public_assets/assets/000/244/965/original/domain_step3_idle.png?1791359473)

Idle shutdown stops a JupyterLab space after it has been idle for the set time. The default the box offers is 10,080 minutes, which is seven days of billing for a forgotten space. 60 minutes is the smallest value it accepts. This setting exists because people forget to stop spaces.

![domain_step5_network.png](https://d2beiqkhq929f0.cloudfront.net/public_assets/assets/000/244/966/original/domain_step5_network.png?1791359475)

![domain_step5_subnets.png](https://d2beiqkhq929f0.cloudfront.net/public_assets/assets/000/244/967/original/domain_step5_subnets.png?1791359478)

Step 5 checks the VPC for you. The yellow box lists the Canvas services it cannot reach, and the only one is Amazon Bedrock. That powers Canvas's generative AI features, which these classes do not use.

![domain_step6_storage.png](https://d2beiqkhq929f0.cloudfront.net/public_assets/assets/000/244/760/original/domain_step6_storage.png?1791291698)

The EFS file system is a shared home folder for every user in the domain. Spaces keep their files on their own disk, so these classes do not need it. In this run the box was already off.

![domain_created.png](https://d2beiqkhq929f0.cloudfront.net/public_assets/assets/000/244/961/original/domain_created.png?1791359461)

![domain_network.png](https://d2beiqkhq929f0.cloudfront.net/public_assets/assets/000/244/962/original/domain_network.png?1791359464)

The domain was ready in under half a minute. Its **Network** section shows **VPC only**, your VPC, both subnets and the studio group, and the role is `returns-ml-sagemaker-role`.

### 5.3 A space with remote access

**[Instructor note]** Browser. On the SageMaker Studio page, pick the `instructor` profile and click **Open Studio**. Studio opens in a new tab with a welcome tour. Click **Skip Tour**. Then click **JupyterLab** under Applications, then **Create JupyterLab space**. Name it `returns-lab`, keep it **Private** and click **Create space**.

**User profile to open**

In [ ]:
instructor

**JupyterLab space name**

In [ ]:
returns-lab

![studio_tour.png](https://d2beiqkhq929f0.cloudfront.net/public_assets/assets/000/244/993/original/studio_tour.png?1791359537)

![space_create.png](https://d2beiqkhq929f0.cloudfront.net/public_assets/assets/000/244/513/original/space_create.png?1791270282)

The new space starts on `ml.t3.medium`, which has 4 GB of memory. Remote access needs at least 8 GB, and the page does not warn you. You pick a bigger machine yourself.

**[Instructor note]** Browser, in the Studio tab. Open the **Instance** box and hover over **Fast launch**, not General Purpose. Pick `ml.m5.large`. The General Purpose list starts at `ml.m5.xlarge`, which costs twice as much. Then turn on **Remote access** further down the page and click **Run space**.

**Instance type to select under Fast launch**

In [ ]:
ml.m5.large

![space_instance_menu.png](https://d2beiqkhq929f0.cloudfront.net/public_assets/assets/000/244/988/original/space_instance_menu.png?1791359524)

![space_settings.png](https://d2beiqkhq929f0.cloudfront.net/public_assets/assets/000/244/990/original/space_settings.png?1791359529)

![space_running.png](https://d2beiqkhq929f0.cloudfront.net/public_assets/assets/000/244/989/original/space_running.png?1791359527)

The space was running 39 seconds after **Run space**. The **Open in VS Code**, **Open in Kiro** and **Open in Cursor** buttons are now active. **Idle Shutdown** shows 60 and is greyed out, because the domain set it.

### 5.4 Setting up your laptop

VS Code on your laptop needs a few tools before it can connect to a space. This list is from AWS's remote access guide.

| What | Why it is needed | Check it with |
|---|---|---|
| VS Code 1.90 or newer | The editor itself. Use Microsoft's official build | `code --version` |
| The `code` command | AWS lists it as required, and the checks below use it | `code --version` |
| AWS Toolkit extension | AWS lists it as required for every way of connecting | `code --list-extensions` |
| Remote - SSH extension | The connection is SSH carried over a Systems Manager tunnel | `code --list-extensions` |
| Session Manager plugin | Opens that tunnel from your side | `session-manager-plugin --version` |
| An SSH client | Carries the SSH part | `ssh -V` |

**Optional complete extension list — laptop terminal**

In [ ]:
code --list-extensions

AWS supports macOS 13 or newer, Windows 10 or 11, and Linux. When you connect, the Toolkit checks for these tools and offers to install any that are missing. Do not rely on that in class. A company laptop often blocks installs, and the `code` command is often missing on a Mac. Go through the steps below once before class. Each step checks first, so anything you already have is skipped.

The outputs below are from a Mac with Apple silicon. Your version numbers will differ. Anything at or above the minimum is fine.

**[Instructor note]** Laptop, before class, in a terminal on your laptop (Terminal on a Mac, PowerShell on Windows). Run each check. Only run the install lines for a check that fails.

**Step 1. VS Code and the `code` command.** Run this.

**Copy command → laptop terminal**

In [ ]:
code --version


```
1.140.0
07f806f999227108933c2e30515b26eecc1fda74
arm64
```

The first line is the VS Code version, and it must be 1.90 or higher. If you see `command not found`, there are two possible reasons.

- **VS Code is not installed.** Download it from [code.visualstudio.com](https://code.visualstudio.com/) and install it. On a Mac, drag it into the Applications folder. On a Mac with Homebrew, `brew install --cask visual-studio-code` does the same.

**Install VS Code with Homebrew, if needed — macOS terminal**

In [ ]:
brew install --cask visual-studio-code

- **VS Code is installed but `code` is not on your PATH.** This is the usual case on a Mac. Open VS Code, open the Command Palette with Cmd+Shift+P, type `shell command` and pick **Shell Command: Install 'code' command in PATH**. On Windows and Linux the installer adds it for you. On Windows, tick **Add to PATH** during install if it asks.

**VS Code Command Palette search**

In [ ]:
shell command

Close the terminal, open a new one, and run `code --version` again. It must print a version now.

**Recheck VS Code — new laptop terminal**

In [ ]:
code --version

**Step 2. The two extensions.** Run this.

**Copy extension check → macOS or Linux terminal**

In [ ]:
code --list-extensions | grep -i -E "aws-toolkit|remote-ssh"


```
amazonwebservices.aws-toolkit-vscode
ms-vscode-remote.remote-ssh
ms-vscode-remote.remote-ssh-edit
```

You need the first two lines. The third comes along with Remote - SSH. On Windows PowerShell, use `code --list-extensions | Select-String "aws-toolkit|remote-ssh"` instead. If a line is missing, install that extension by its ID.

**Extension check — Windows PowerShell**

In [ ]:
code --list-extensions | Select-String "aws-toolkit|remote-ssh"

**Copy installation commands → laptop terminal, if an extension is missing**

In [ ]:
code --install-extension amazonwebservices.aws-toolkit-vscode
code --install-extension ms-vscode-remote.remote-ssh


Running an install for an extension you already have is harmless. It only says so.

```
Installing extensions...
Extension 'amazonwebservices.aws-toolkit-vscode' v4.17.0 is already installed. Use '--force' option to update to latest version or provide '@<version>' to install a specific version, for example: 'amazonwebservices.aws-toolkit-vscode@1.2.3'.
```

The same works from inside VS Code. Open the **Extensions** view, search for `AWS Toolkit` and `Remote - SSH`, and click **Install** on each. Run the check again until both lines show.

**VS Code Extensions search**

In [ ]:
AWS Toolkit

**VS Code Extensions search**

In [ ]:
Remote - SSH

**Step 3. The Session Manager plugin.** Run this.

**Copy version check → laptop terminal**

In [ ]:
session-manager-plugin --version


```
1.2.835.0
```

Any version number means it is installed. If you see `command not found`, install it the way AWS's [install guide](https://docs.aws.amazon.com/systems-manager/latest/userguide/session-manager-working-with-install-plugin.html) shows for your system. On a Mac with Apple silicon, these are AWS's commands. They ask for your Mac password, because the plugin goes into a system folder.

**Copy installation commands → Apple silicon Mac terminal, if the plugin is missing**

In [ ]:
curl "https://s3.amazonaws.com/session-manager-downloads/plugin/latest/mac_arm64/session-manager-plugin.pkg" -o "session-manager-plugin.pkg"
sudo installer -pkg session-manager-plugin.pkg -target /
sudo ln -s /usr/local/sessionmanagerplugin/bin/session-manager-plugin /usr/local/bin/session-manager-plugin


On an Intel Mac, change `mac_arm64` to `mac` in the first line. On a Mac with Homebrew, `brew install --cask session-manager-plugin` does all three. On Windows, download and run [SessionManagerPluginSetup.exe](https://s3.amazonaws.com/session-manager-downloads/plugin/latest/windows/SessionManagerPluginSetup.exe), which needs administrator rights, then open a new PowerShell window. Run the check again. Running `session-manager-plugin` with nothing after it is a second check. It prints "The Session Manager plugin was installed successfully".

**Alternative install — Intel Mac terminal, if needed**

In [ ]:
curl "https://s3.amazonaws.com/session-manager-downloads/plugin/latest/mac/session-manager-plugin.pkg" -o "session-manager-plugin.pkg"
sudo installer -pkg session-manager-plugin.pkg -target /
sudo ln -s /usr/local/sessionmanagerplugin/bin/session-manager-plugin /usr/local/bin/session-manager-plugin

**Alternative install with Homebrew — macOS terminal, if needed**

In [ ]:
brew install --cask session-manager-plugin

**Recheck Session Manager — new laptop terminal**

In [ ]:
session-manager-plugin --version

**Second Session Manager check — laptop terminal**

In [ ]:
session-manager-plugin

**Step 4. The SSH client.** Run this.

**Copy SSH check → laptop terminal**

In [ ]:
ssh -V


```
OpenSSH_10.3p1, LibreSSL 3.3.6
```

Macs already have one, and so do most Linux systems. On Windows, if PowerShell says `ssh` is not recognised, search Settings for **Optional features**, add a feature, pick **OpenSSH Client** and install it. Then open a new PowerShell window and run the check again.

**Windows Optional features search, if SSH is missing**

In [ ]:
OpenSSH Client

When all four checks print a version, the laptop is ready. Open VS Code once and let it finish any extension updates, so that it does not stop to update in the middle of class.

### 5.5 Connecting VS Code

The connection does not open any port to the internet. VS Code on your laptop asks AWS to start a session on the space, AWS Systems Manager carries the traffic through the `ssm` and `ssmmessages` endpoints from 4.3, and VS Code runs a small server on the space over that tunnel. From then on the VS Code window looks local, but its files, terminal and Python run on the space, inside the private network.

**[Instructor note]** Laptop. On the space page in Studio, click **Open in VS Code** and allow the browser to open VS Code. VS Code asks once more to open the remote window. Accept, and wait while it installs its server on the space. That download goes out through the NAT gateway. When the window is ready, open a terminal in it with **Terminal → New Terminal**. That terminal is on the space, so the prompt shows `sagemaker-user`. Run `mkdir -p ~/returns-ml && cd ~/returns-ml`, then **File → Open Folder** and open `/home/sagemaker-user/returns-ml`. Every file from here on is created in that folder from VS Code's file explorer.

**Create and enter the project folder — VS Code terminal connected to the SageMaker space**

In [ ]:
mkdir -p ~/returns-ml && cd ~/returns-ml

**File → Open Folder — path on the SageMaker space**

In [ ]:
/home/sagemaker-user/returns-ml

The space has no access keys in it, and the code below never sets any. It runs as `returns-ml-sagemaker-role`, which is why it can read your bucket. Its requests to AWS go out through the endpoints from 4.3. The image already has pandas, scikit-learn, boto3 and the SageMaker Python SDK installed.

## **6. A first look at the data**

Every file in the project reads the bucket name from one place.

**[Instructor note]** VS Code, new file `returns-ml/config.py`. Change `BUCKET` to your own bucket name.

**New file name — inside the returns-ml folder**

In [ ]:
config.py

**Copy the complete file contents → config.py in VS Code**

In [ ]:
"""Settings every script in this folder shares. Change BUCKET to your own bucket."""
BUCKET = "returns-ml-demo-1007"
REGION = "us-east-1"

`boto3` is AWS's Python library. `boto3.client("s3")` gives you an object with one method per S3 action, and `get_object` downloads one object. Pandas reads the downloaded body directly, so nothing is saved to disk. The second half of this file is a check for a problem in the data, explained after the output.

**[Instructor note]** VS Code, new file `returns-ml/explore.py`. Run it from VS Code's terminal with `python explore.py`.

**New file name — inside the returns-ml folder**

In [ ]:
explore.py

**Copy the complete file contents → explore.py in VS Code**

In [ ]:
"""Load the raw orders from S3 and check for a leaky column."""
import boto3
import pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split

from config import BUCKET

s3 = boto3.client("s3")
raw = pd.read_csv(s3.get_object(Bucket=BUCKET, Key="raw/returns.csv")["Body"])

print(raw.shape)
print("return rate:", round(raw["returned"].mean(), 3))
print("missing values:", raw.isna().sum()[lambda s: s > 0].to_dict())
print(raw.head(3).T)


def quick_auc(columns_to_drop):
    X = pd.get_dummies(raw.drop(columns=columns_to_drop), dtype=int)
    y = raw["returned"]
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=0, stratify=y)
    model = HistGradientBoostingClassifier(random_state=0).fit(X_tr, y_tr)
    return round(roc_auc_score(y_te, model.predict_proba(X_te)[:, 1]), 3)


base = ["order_id", "order_date", "returned"]
print("with refund_amount   :", quick_auc(base))
print("without refund_amount:", quick_auc(base + ["refund_amount"]))

**Run after saving the file → VS Code terminal on the SageMaker space**

In [ ]:
python explore.py

Output from the run.

```
(12000, 14)
return rate: 0.228
missing values: {'seller_rating': 502}
                               0            1           2
order_id              ORD-100000   ORD-100001  ORD-100002
order_date            2026-05-05   2026-02-23  2026-07-29
category                 apparel  electronics        home
price                     1257.0       6717.0      1354.0
discount_pct                  18           25          10
delivery_days                  4            4           3
customer_tenure_days        1096          116         256
prior_orders                  13            3           3
prior_returns                  9            1           0
payment_method            wallet          upi        card
is_gift                        0            0           0
seller_rating                4.9          4.0         3.4
refund_amount             1031.0          0.0         0.0
returned                       1            0           0
with refund_amount   : 1.0
without refund_amount: 0.706
```

Twelve thousand orders, and 22.8% of them came back. A model that always says "not returned" would be right 77% of the time, which is why these classes judge models by AUC and not by accuracy. Only `seller_rating` has gaps, 502 of them.

Look at the column list. Most columns describe the order when it was placed. One of them does not. `refund_amount` is filled in after the return has been processed. For an order that was never returned it is always zero. The first order came back, and its refund is 1031.0. The next two did not, and theirs are 0.0.

### 6.1 A column from the future

A column like that is called **target leakage**. It holds information that would not exist yet at the moment you need a prediction. A model will happily learn from it, and its scores will look superb. The last two lines of the output are the quickest check. They train two small models on the space, one with the column and one without, and compare.

An AUC of 1.0 is a perfect score, and on real data a perfect score almost always means a leak. The model did not learn anything about returns. It learned that a refund means a return. In production `refund_amount` is always zero at prediction time, so this model would call every order safe.

The honest number is 0.706. That is a modest model, and modest is normal for return prediction. Whether a customer sends something back depends a lot on things the shop never sees. From here on `refund_amount` is dropped.

### **Checkpoint 6.1**

Pick one other column in this table and decide whether it is known when the order is placed. Try `delivery_days`. Is the number of days the parcel took known at order time, or only an estimate? Write one sentence on what you would need to check with the people who built the data. Reasoning about when each column is known catches leaks that no tool can, because the tool does not know your business.

## **7. Data Wrangler in Canvas**

You found the leak with a few lines of Python. Data Wrangler is SageMaker's point-and-click tool for exploring and cleaning data, and it is worth seeing whether it finds the same thing without any code. It used to live inside the old Studio. It now lives only inside **SageMaker Canvas**, the no-code workspace. You reach Canvas from Studio or from the console.

Remember the cost from section 2. Canvas bills **1.90 USD an hour from the moment it opens until you log out**. New accounts get a two-month free tier on Canvas time, but do not rely on it.

**[Instructor note]**  Browser, in the Studio tab from 5.3. Click **Canvas** under **Applications** in the top left, then click **Open Canvas**. Canvas opens in a new tab. Starting it took 129 seconds in the recorded run. In Canvas, click **Data Wrangler** in the left bar.

![canvas_home.png](https://d2beiqkhq929f0.cloudfront.net/public_assets/assets/000/244/959/original/canvas_home.png?1791359455)

![dw_home.png](https://d2beiqkhq929f0.cloudfront.net/public_assets/assets/000/244/787/original/dw_home.png?1791291758)

**[Instructor note]** Browser. Click **Import and prepare**, then **Tabular**. Change the data source from Canvas Datasets to **Amazon S3**. Open your bucket, then `raw`, tick `returns.csv`, click **Next** and then **Import**.

![dw_source.png](https://d2beiqkhq929f0.cloudfront.net/public_assets/assets/000/244/974/original/dw_source.png?1791359492)

The list of data sources came up in 13 seconds. In the closed network described in 4.4, the same list never loaded.

![dw_pick_s3.png](https://d2beiqkhq929f0.cloudfront.net/public_assets/assets/000/244/971/original/dw_pick_s3.png?1791359485)

![dw_preview.png](https://d2beiqkhq929f0.cloudfront.net/public_assets/assets/000/244/494/original/dw_preview.png?1791270241)

The import opens a **data flow**, a chain of steps starting from the source. It has already added a step that guesses each column's type. Click the plus sign after that step to see what you can add.

![dw_flow_menu.png](https://d2beiqkhq929f0.cloudfront.net/public_assets/assets/000/244/969/original/dw_flow_menu.png?1791359481)

**[Instructor note]** Browser. Click **Get data insights**. Set the target column to `returned`, the problem type to **Classification**, and click **Create**. The report took 43 seconds here.

**Target column**

In [ ]:
returned

![dw_insights_form.png](https://d2beiqkhq929f0.cloudfront.net/public_assets/assets/000/244/970/original/dw_insights_form.png?1791359483)

![dw_report_summary.png](https://d2beiqkhq929f0.cloudfront.net/public_assets/assets/000/244/973/original/dw_report_summary.png?1791359489)

The summary agrees with what `explore.py` printed. There are 12,000 rows, 14 columns and 0.322% of values missing, which is the 502 missing seller ratings. Scroll down to the warnings.

![dw_report_leakage.png](https://d2beiqkhq929f0.cloudfront.net/public_assets/assets/000/244/972/original/dw_report_leakage.png?1791359487)

Data Wrangler found the same leak as `explore.py`. It flags `refund_amount` as **Target leakage, High**, and even suggests the fix, a Drop column transform. The feature table below it scores each column alone.

![dw_feature_power.png](https://d2beiqkhq929f0.cloudfront.net/public_assets/assets/000/244/968/original/dw_feature_power.png?1791359479)

`refund_amount` scores 0.583 on its own, more than twice the next column, `price`, at 0.271. That gap is what triggered the warning. It checks the same question you answered with two quick models in 6.1, without any code.

### 7.1 Where Data Wrangler fits

That warning is useful, and an analyst who does not write Python gets it in a few clicks. But the work you do in the flow lives inside Canvas. In a team of engineers, the cleaning steps live in code that goes through Git, review and CI, the way `prepare.py` in the next section does. Data Wrangler can export a flow as a Python script or a processing job, but a generated script is not the same as code your team owns. A fair way to use it is to look at a new dataset fast, find the problems, and then write the fix where the rest of your code lives. Weigh that against 1.90 USD an hour.

**[Instructor note]** Browser. Click **Log out** at the bottom of the Canvas left bar, then **Log out** in the dialog. Read the dialog out loud. It says that logging out is what stops the workspace charges.

![canvas_logout.png](https://d2beiqkhq929f0.cloudfront.net/public_assets/assets/000/244/960/original/canvas_logout.png?1791359459)

After logging out, the Canvas app was gone from the domain's app list. Closing the tab does not do this.

## **8. Getting the data into the shape XGBoost expects**

SageMaker comes with **built-in algorithms**. These are ready-made training containers that AWS maintains, and XGBoost is the one most used for tables. The next class trains with it. You do not write any training code for it. The price is that you must hand over the data in exactly its format. For CSV that means three things.

- **The label goes in the first column.**
- **There is no header row.**
- **Every value is a number.** Categories such as `apparel` must already be turned into columns of 0s and 1s.

**[Instructor note]** VS Code, new file `returns-ml/prepare.py`. Run it with `python prepare.py`. It writes four files to your bucket.

**New file name — inside the returns-ml folder**

In [ ]:
prepare.py

**Copy the complete file contents → prepare.py in VS Code**

In [ ]:
"""Write train, validation and batch files in the format the built-in XGBoost expects."""
import boto3
import pandas as pd

from config import BUCKET

s3 = boto3.client("s3")
raw = pd.read_csv(s3.get_object(Bucket=BUCKET, Key="raw/returns.csv")["Body"])

# The built-in XGBoost reads CSV with the label in the FIRST column and NO header row.
features = raw.drop(columns=["order_id", "order_date", "refund_amount", "returned"])
features = pd.get_dummies(features, columns=["category", "payment_method"], dtype=int)
data = pd.concat([raw["returned"], features], axis=1).sample(frac=1, random_state=0)

n = len(data)
train = data.iloc[: int(n * 0.7)]
validation = data.iloc[int(n * 0.7): int(n * 0.85)]
batch = data.iloc[int(n * 0.85):]


def put(df, key, header=False):
    s3.put_object(Bucket=BUCKET, Key=key, Body=df.to_csv(header=header, index=False))


put(train, "train/train.csv")
put(validation, "validation/validation.csv")
put(batch.drop(columns=["returned"]), "batch-input/batch.csv")   # features only
put(batch[["returned"]], "batch-labels/labels.csv", header=True)  # kept aside for scoring

print("train", train.shape, "validation", validation.shape, "batch", batch.shape)
print("feature order:", list(features.columns))
first = s3.get_object(Bucket=BUCKET, Key="train/train.csv")["Body"].read().decode().splitlines()[0]
print("first line of train.csv:", first)

**Run after saving the file → VS Code terminal on the SageMaker space**

In [ ]:
python prepare.py

Output from the run.

```
train (8400, 19) validation (1800, 19) batch (1800, 19)
feature order: ['price', 'discount_pct', 'delivery_days', 'customer_tenure_days', 'prior_orders', 'prior_returns', 'is_gift', 'seller_rating', 'category_apparel', 'category_beauty', 'category_books', 'category_electronics', 'category_footwear', 'category_home', 'payment_method_card', 'payment_method_cod', 'payment_method_upi', 'payment_method_wallet']
first line of train.csv: 1,608.0,16,5,735,11,3,1,3.7,0,1,0,0,0,0,0,0,0,1
```

The first line of `train.csv` starts with the label, `1`, and has no header. The 18 columns after it are the features, in the order printed above. Keep that order in mind. The endpoint you build next class expects exactly these 18 numbers in exactly this order, and nothing checks it for you.

The split is 70% for training, 15% for validation during training, and 15% kept back as a batch to score at the end. The batch file has no label column, because a model being asked for predictions should not be given the answers. The answers go to `batch-labels/` so the batch scores can be checked next class.

### 8.1 What a header row does

The obvious mistake is to forget `header=False`. You would expect an error, because the first line would be words, not numbers. While preparing these classes, a training job was run on a copy of the file with the header left in. It did not fail. Its log said this.

```
Train matrix has 9601 rows and 18 columns
```

There were only 9,600 training rows in that run, so the header was read as one extra row of data, with its words turned into missing values. The job finished and the model was nearly identical. One wrong row out of nine thousand barely matters. The lesson is that the built-in algorithm does not check your format. It reads whatever it is given. The row count in the log is the only clue, so read it.

## **9. Shutting down until next class**

Four things from this class bill by the hour. Those are Canvas, the space, the NAT gateway and the interface endpoints. Canvas is already done. The rest are stopped or deleted here. The bucket, the role, the domain and the VPC itself cost nothing or almost nothing, and the next class needs them, so they stay.

**[Instructor note]** Laptop and browser. Close the remote VS Code window. Then in the Studio tab open **JupyterLab**, open `returns-lab` and click **Stop space**, then **Stop space** in the dialog.

**JupyterLab space to stop**

In [ ]:
returns-lab

![space_stop_dialog.png](https://d2beiqkhq929f0.cloudfront.net/public_assets/assets/000/244/991/original/space_stop_dialog.png?1791359532)

The blue box in that dialog is worth reading out loud. Stopping the space does **not** delete endpoints, buckets or anything else you created from it. It only stops the space's own machine. Your files in `returns-ml` stay on the space's disk and are there again next class.

![space_stopping.png](https://d2beiqkhq929f0.cloudfront.net/public_assets/assets/000/244/992/original/space_stopping.png?1791359534)

**[Instructor note]** Browser. In the VPC console open **Endpoints**. Tick every endpoint whose type is **Interface**, which is all of them except `returns-ml-vpce-s3`. Choose **Actions → Delete VPC endpoints**, type `delete` and click **Delete**.

**S3 gateway endpoint to keep**

In [ ]:
returns-ml-vpce-s3

**Delete VPC endpoints confirmation — after selecting the interface endpoints described above**

In [ ]:
delete

![vpce_delete.png](https://d2beiqkhq929f0.cloudfront.net/public_assets/assets/000/245/002/original/vpce_delete.png?1791359556)

**[Instructor note]** Browser. Open **NAT gateways**, select `returns-ml-regional-nat`, choose **Actions → Delete NAT gateway**, type `delete` and click **Delete**.

**NAT gateway to select**

In [ ]:
returns-ml-regional-nat

**Delete NAT gateway confirmation**

In [ ]:
delete

![nat_delete.png](https://d2beiqkhq929f0.cloudfront.net/public_assets/assets/000/244/976/original/nat_delete.png?1791359496)

The NAT gateway had a public IP address that AWS gave it automatically. After the NAT gateway was deleted, that address stayed in **Elastic IPs** for a few minutes and then disappeared by itself. If you ever give a NAT gateway an address of your own, you have to release it yourself, because an unused address also bills by the hour. Check **Elastic IPs** once before you leave.

Deleting these two does leave a mark. The private route tables still have their `0.0.0.0/0` route, now pointing at a NAT gateway that no longer exists. AWS shows such a route as a **blackhole**. Nothing is broken by it while the space is stopped, and the next class starts by pointing it at a new NAT gateway.

What is left now, and what it costs between classes.

| Left behind | Costs | Needed next class? |
|---|---|---|
| The space, stopped | Its 5 GB disk, a few cents a month | Yes, it holds your files |
| The domain and user profile | Nothing | Yes |
| The VPC, its subnets, security groups and S3 endpoint | Nothing | Yes |
| The role | Nothing | Yes |
| The bucket | About 1 MB of storage | Yes, it holds the prepared data |

Doing it this way is a trade. Leaving the NAT gateway and endpoints up for three days until the next class would cost about 12 US dollars and teach the habit of leaving things running. Deleting them costs a few minutes of setup next class, done with one script.

## **10. Class summary**

The work split into two jobs. As the platform engineer you built a private network, a role and a VPC-only domain in the console. The network had no way in from the internet. It had VPC endpoints for the AWS services that carry data and control traffic, and a NAT gateway as a one-way door out for everything else. As the ML engineer you connected VS Code to a space inside that network and did the data work in Python files, which is where this kind of work lives in a real team.

The data had one trap. The leaky `refund_amount` column scored a perfect AUC until it was taken out, and two quick models side by side showed it in seconds. Data Wrangler found the same column without code, which is useful for a first look and costly to leave open. Then `prepare.py` turned the raw export into the strict format the built-in XGBoost algorithm wants, with the label first, no header and only numbers, and the algorithm will not tell you if you get that wrong.

The quiet failures were the lessons. VS Code would not connect until the role had one permission that no message named. A closed network left a page loading forever when an endpoint was missing. A header row would have been read as data with nothing but a row count to show it. None of these raised a clear error, and catching them came down to reading the numbers and the logs.

## **11. References and further reading**

**Official documentation**

- [Amazon SageMaker AI developer guide](https://docs.aws.amazon.com/sagemaker/latest/dg/whatis.html). Start with the pages on domains and spaces.
- [Connect your Remote IDE to SageMaker spaces](https://docs.aws.amazon.com/sagemaker/latest/dg/remote-access.html). Setting up VS Code remote access, the permissions it needs, and what changes in a network with no internet.
- [Connect Studio in a VPC to external resources](https://docs.aws.amazon.com/sagemaker/latest/dg/studio-updated-and-internet-access.html). The rules for a VPC only domain, including the security group ports and the endpoints AWS asks for.
- [Configure Canvas in a VPC without internet access](https://docs.aws.amazon.com/sagemaker/latest/dg/canvas-vpc.html). AWS's list of endpoints for Canvas. Section 4.4 needed one that is not on it.
- [XGBoost algorithm with Amazon SageMaker AI](https://docs.aws.amazon.com/sagemaker/latest/dg/xgboost.html). The input format rules that `prepare.py` follows.
- [NAT gateways](https://docs.aws.amazon.com/vpc/latest/userguide/vpc-nat-gateway.html). Zonal and regional NAT gateways and how private subnets route through them.
- [AmazonSageMakerFullAccess policy reference](https://docs.aws.amazon.com/aws-managed-policy/latest/reference/AmazonSageMakerFullAccess.html). Its S3 statements show why it only covers buckets with `sagemaker` in the name.

**Pricing**

- [Amazon SageMaker AI pricing](https://aws.amazon.com/sagemaker/ai/pricing/), [Canvas pricing](https://aws.amazon.com/sagemaker/ai/canvas/pricing/), [Amazon VPC pricing](https://aws.amazon.com/vpc/pricing/) for NAT gateways, and [AWS PrivateLink pricing](https://aws.amazon.com/privatelink/pricing/) for endpoints. Check these before relying on the numbers in section 2, because prices change.

**Practitioner reading**

- [Leakage in data mining, Kaufman, Rosset, Perlich and Stitelman (2012)](https://dl.acm.org/doi/10.1145/2382577.2382579). The standard paper on target leakage, with real competition examples.

## **12. What's next**

**Before the next class**

1. Check **Endpoints**, **NAT gateways** and **Elastic IPs** in the VPC console and make sure none of them has anything that bills. Then check that your space shows **Stopped** and that you logged out of Canvas.
2. Open your bucket in S3 and check that `train/`, `validation/`, `batch-input/` and `batch-labels/` each hold one file. The next class starts from them.
3. Do Checkpoint 6.1 if you did not get to it in class.
4. Look at your AWS bill under **Billing and Cost Management** tomorrow and find the SageMaker and VPC lines. Seeing your own few cents is the best way to remember what bills and what does not.

**Where this goes**

The next class opens the same VS Code window and turns these four files into a model. A training job runs on a rented machine, and the model is served three ways, as a real-time endpoint, a serverless endpoint and a batch job. The network you built by hand is what a team's infrastructure code builds for them, so every project gets the same safe setup.